# Clinical Drug Monograph RAG — TNF Blocker Q&A

This notebook demonstrates a Retrieval-Augmented Generation (RAG) pipeline applied to a clinical drug monograph (a TNF-alpha inhibitor). Given a PDF monograph, it:

1. Loads and chunks the document
2. Embeds chunks using Gemini's `gemini-embedding-001` model with document-optimized task type
3. Stores embeddings in a persistent ChromaDB vector store
4. Answers clinical questions (dosing, side effects, contraindications) by retrieving relevant chunks and passing them to Gemini

**Use case:** Rapid Q&A over dense clinical reference documents — relevant to pharmacovigilance, formulary review, and clinical decision support.

**Stack:** Google Gemini (embeddings + generation), ChromaDB, LangChain, PyPDF, Google Cloud Storage

## 1. Install Dependencies

In [ ]:
!pip install -q chromadb google-genai langchain-community langchain-text-splitters langchain-google-community[gcs] pypdf

## 2. Authenticate with Google Cloud

This notebook runs in Google Colab and uses Application Default Credentials.

In [ ]:
from google.colab import auth
auth.authenticate_user()

## 3. Configuration

Replace the placeholders below with your own GCP project ID and Gemini API key.
Never commit real credentials to version control — use environment variables or Colab Secrets in production.

In [ ]:
# Replace with your values
GEMINI_API_KEY = "your-gemini-api-key"   # or use: from google.colab import userdata; userdata.get('GEMINI_API_KEY')
GCP_PROJECT_ID = "your-gcp-project-id"
GCS_BUCKET    = "your-gcs-bucket"
GCS_BLOB      = "path/to/your-monograph.pdf"

## 4. Imports

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import google.generativeai as genai
from google import genai as genai_client
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_community import GCSFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List

## 5. Gemini Embedding Function

We define a custom ChromaDB embedding function backed by Gemini.
Note the `task_type` distinction: `retrieval_document` for indexing, `retrieval_query` for querying — this improves retrieval quality.

In [ ]:
genai.configure(api_key=GEMINI_API_KEY)

class GeminiEmbeddingFunction(embedding_functions.EmbeddingFunction):
    """ChromaDB-compatible embedding function using Gemini embedding-001."""

    def __call__(self, input: List[str]) -> List[List[float]]:
        response = genai.embed_content(
            model="gemini-embedding-001",
            content=input,
            task_type="retrieval_document"  # optimized for indexing
        )
        if isinstance(response, dict):
            return response["embedding"]
        return response.embedding

embedding_function = GeminiEmbeddingFunction()

## 6. Initialize ChromaDB Vector Store

In [ ]:
client_db = chromadb.PersistentClient(path=".")
collection = client_db.get_or_create_collection(
    name="chat-with-pdf",
    embedding_function=embedding_function
)
print(f"Collection document count: {collection.count()}")

## 7. Load, Chunk, and Embed the Monograph

The monograph is loaded from Google Cloud Storage, split into 1,000-character chunks with 100-character overlap, and embedded into ChromaDB.

> **Note:** Embedding a full monograph takes a few minutes depending on document length.

In [ ]:
def load_pdf(file_path):
    return PyPDFLoader(file_path)

loader = GCSFileLoader(
    project_name=GCP_PROJECT_ID,
    bucket=GCS_BUCKET,
    blob=GCS_BLOB,
    loader_func=load_pdf
)

pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

documents = [chunk.page_content for chunk in chunks]
ids = [f"doc_{i}" for i in range(len(documents))]

collection.add(documents=documents, ids=ids)
print(f"Indexed {len(documents)} chunks. Collection size: {collection.count()}")

## 8. RAG Query Function

For each query:
1. Embed the question using `task_type="retrieval_query"` (query-optimized)
2. Retrieve the top-k most relevant chunks from ChromaDB
3. Truncate context to a character budget (cost control)
4. Pass context + question to Gemini with a grounded prompt that restricts answers to the retrieved context

In [ ]:
gemini = genai_client.Client(api_key=GEMINI_API_KEY)

def rag_query(query: str, collection, top_k: int = 5, max_context_chars: int = 6000) -> str:
    """Answer a clinical question using RAG over the embedded monograph."""

    # 1. Embed query (retrieval-optimized)
    response = gemini.models.embed_content(
        model="gemini-embedding-001",
        contents=query,
        config={"task_type": "retrieval_query"}
    )
    query_embedding = response.embeddings[0].values

    # 2. Retrieve top-k chunks
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    documents = results["documents"][0]

    # 3. Build context with character budget
    context_parts, total_chars = [], 0
    for doc in documents:
        if total_chars + len(doc) > max_context_chars:
            break
        context_parts.append(doc)
        total_chars += len(doc)
    context = "\n\n".join(context_parts)

    # 4. Grounded prompt — answers restricted to retrieved context
    prompt = f"""You are a helpful clinical assistant.
Answer ONLY using the provided context from the drug monograph.
If the answer is not in the context, say \"I don't know.\"

Context:
{context}

Question:
{query}

Answer:"""

    # 5. Generate answer
    response = gemini.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text

## 9. Sample Clinical Queries

Demonstrating Q&A over the TNF-alpha inhibitor monograph:

In [ ]:
queries = [
    "What is the initiation dose for this TNF blocker?",
    "What are the potential side effects of this medication?",
    "What is the most severe side effect?",
    "What doses are available for this medication?",
    "What are the contraindications?",
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {rag_query(q, collection, top_k=3, max_context_chars=4000)}")
    print("-" * 60)

## 10. Next Steps

- **Embedding strategy:** Consider domain-specific biomedical embedding models (e.g. MedCPT, BiomedBERT) for clinical terminology — Gemini embeddings are general-purpose
- **Chunking by page:** Splitting by page before applying character chunking reduces the risk of splitting mid-sentence across clinical sections
- **Multi-document:** Extend to a formulary or full drug class (all TNF inhibitors) for comparative Q&A
- **Evaluation:** Add a gold-standard Q&A set to measure retrieval precision and answer accuracy